# Phase 5E — Feature Engineering & sklearn Pipelines

Feature engineering is often the most impactful step in ML. Garbage in → garbage out.

**Topics:** Encoding, scaling, imputation, feature selection, sklearn Pipeline.

**Install:** `pip install scikit-learn`

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    PolynomialFeatures,
)
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

np.random.seed(42)
sns.set_theme(style="whitegrid")

---
## 1. Scaling — When and What to Use

In [ ]:
data = np.array(
    [[1000, 5, 0.001], [2000, 8, 0.003], [1500, 3, 0.002], [3000, 10, 0.005]]
)
df_scale = pd.DataFrame(data, columns=["income", "years_exp", "click_rate"])

scalers = {
    "StandardScaler\n(mean=0, std=1)": StandardScaler(),
    "MinMaxScaler\n(range [0,1])": MinMaxScaler(),
    "RobustScaler\n(uses IQR, robust to outliers)": RobustScaler(),
}

fig, axes = plt.subplots(1, len(scalers) + 1, figsize=(15, 4))

axes[0].bar(df_scale.columns, df_scale.max(), color="coral", label="Max value")
axes[0].set_title("Original\n(very different scales!)")
axes[0].set_yscale("log")

for ax, (name, scaler) in zip(axes[1:], scalers.items()):
    scaled = scaler.fit_transform(df_scale)
    ax.bar(df_scale.columns, np.abs(scaled).max(axis=0), color="steelblue")
    ax.set_title(name)

plt.tight_layout()
plt.show()

print("When to use each:")
print("  StandardScaler : Default choice. Assumes normal distribution.")
print("  MinMaxScaler   : When you need [0,1] range (e.g. neural networks).")
print("  RobustScaler   : When data has significant outliers.")

---
## 2. Encoding Categorical Variables

In [ ]:
df_cat = pd.DataFrame(
    {
        "color": ["red", "blue", "green", "red", "blue"],
        "size": ["S", "M", "L", "XL", "M"],
        "quality": ["low", "medium", "high", "medium", "high"],
        "price": [10, 25, 30, 20, 22],
    }
)

# One-Hot Encoding — for NOMINAL categories (no order)
ohe = OneHotEncoder(sparse_output=False)
color_encoded = ohe.fit_transform(df_cat[["color"]])
print("One-Hot Encoded 'color':")
print(pd.DataFrame(color_encoded, columns=ohe.get_feature_names_out()))

# Ordinal Encoding — for ORDINAL categories (have meaningful order)
size_order = [["S", "M", "L", "XL"]]
quality_order = [["low", "medium", "high"]]

size_enc = OrdinalEncoder(categories=size_order)
qual_enc = OrdinalEncoder(categories=quality_order)

df_cat["size_encoded"] = size_enc.fit_transform(df_cat[["size"]])
df_cat["quality_encoded"] = qual_enc.fit_transform(df_cat[["quality"]])
print("\nOrdinal Encoded:")
print(df_cat[["size", "size_encoded", "quality", "quality_encoded"]])

---
## 3. sklearn Pipeline — The Right Way

A Pipeline chains preprocessing and modeling into a single object. This prevents **data leakage** and makes deployment clean.

In [ ]:
# Use Titanic dataset
titanic = sns.load_dataset("titanic")
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df_model = titanic[features + ["survived"]].dropna(subset=["survived"])

X = df_model[features]
y = df_model["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identify column types
numerical_cols = ["age", "sibsp", "parch", "fare"]
categorical_cols = ["pclass", "sex", "embarked"]

# Build per-column pipelines
num_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numerical_cols),
        ("cat", cat_pipeline, categorical_cols),
    ]
)

# Full pipeline: preprocess + model
full_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]
)

# Fit on training data
full_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = full_pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Cross-validation on raw X (pipeline handles splits correctly)
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring="accuracy")
print(f"5-fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## 4. Feature Selection

In [ ]:
from sklearn.datasets import make_classification

# Dataset with 20 features, only 5 are truly informative
X_fs, y_fs = make_classification(
    n_samples=500, n_features=20, n_informative=5, n_redundant=5, random_state=42
)

# ANOVA F-score — linear relationship with target
selector_f = SelectKBest(score_func=f_classif, k=5)
selector_f.fit(X_fs, y_fs)

# Mutual Information — captures non-linear relationships too
selector_mi = SelectKBest(score_func=mutual_info_classif, k=5)
selector_mi.fit(X_fs, y_fs)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, selector, title in zip(
    axes, [selector_f, selector_mi], ["ANOVA F-score", "Mutual Information"]
):
    scores = selector.scores_
    selected = selector.get_support(indices=True)
    colors = ["coral" if i in selected else "steelblue" for i in range(20)]
    ax.bar(range(20), scores, color=colors)
    ax.set_xlabel("Feature Index")
    ax.set_ylabel("Score")
    ax.set_title(f"{title} — red = selected top-5")

plt.tight_layout()
plt.show()

---
## Summary — The Pipeline Pattern

```python
Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer()), ('encoder', OneHotEncoder())]),  cat_cols),
    ])),
    ('model', RandomForestClassifier())
])
```

This pattern:
- Prevents data leakage (fit only on train)
- Makes cross-validation correct
- Makes deployment simple (one `predict()` call on raw data)
- Is easy to tune with `GridSearchCV`